Author: Krish

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from datetime import date

In [2]:
data_path = "C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Spring 2025\\STAT390\\LegalAid\\Data\\CAR_-_EP_Flow_Activity_Queue__Agent_Names\\"
adhoc_data_path = 'C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Adhoc\\Adhoc datasets\\'

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
print("Data files read = ",i)

Data files read =  53


In [4]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [5]:
def custdata(id):
    return df_main.loc[df_main['Contact Session ID'] == id,:]

In [6]:
df_main.shape

(3328626, 8)

In [7]:
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [8]:
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

In [9]:
df_main.sort_values(by = ['Contact Session ID', 'Activity Start Timestamp'], inplace=True)

In [10]:
df_main.reset_index(inplace = True, drop = True)

In [11]:
df_main['Date'] = df_main['Activity Start Timestamp'].dt.date

In [13]:
df = df_main.copy()

In [14]:
import pandas as pd

# --- Step 3: Group by Contact Session ID and aggregate ---
grouped = (
    df.groupby('Contact Session ID')
    .agg(
        Call_Start_Time=('Activity Start Timestamp', 'min'),   # earliest timestamp per call
        Starting_Hour=('hour', 'min'),                         # earliest hour per call
        Count=('Activity Start Timestamp', lambda x: len(set(x))),  # number of unique activity timestamps
        Call_Duration=('Activity Start Timestamp', 
                       lambda x: (max(x) - min(x)).total_seconds() / 60 if len(x) > 1 else 0)
    )
    .reset_index()
)

# --- Step 4: Add columns about day specifically  ---
grouped['DayOfWeekNum'] = grouped['Call_Start_Time'].dt.dayofweek + 1     # 1 = Monday, 7 = Sunday
grouped['Call_Start_Date'] = grouped['Call_Start_Time'].dt.date            # date only (no time)

# --- Step 5: Save for Power BI ---
grouped.to_csv(adhoc_data_path + "combined_calls_transformed_simple.csv", index=False)

display(grouped.head())

,Contact Session ID,Call_Start_Time,Starting_Hour,Count,Call_Duration,DayOfWeekNum,Call_Start_Date
0,00002422-f51f-458b-82d6-cfa5a3f36fd9,2025-03-13 12:51:21,12,4,0.900000,4,2025-03-13
1,0000a8d5-cecb-46b1-82cf-b7ce07d85b24,2025-03-17 16:53:15,16,5,0.933333,1,2025-03-17
2,00011655-35de-476f-9a8c-dd48ed4d914a,2024-11-06 14:39:01,14,10,4.133333,3,2024-11-06
3,00014a58-a6ce-4cb2-a529-d55e2c9c304d,2025-02-28 08:33:40,8,8,2.283333,5,2025-02-28
4,00015327-f646-462f-a585-0552331eed4e,2025-06-03 07:43:56,7,3,0.200000,2,2025-06-03
